# Método de gradiente conjugado.

**Objetivo.** 

Describir e implementar el algoritmo de gradiente conjugado más rápido para la solución de sistemas de ecuaciones lineales.

**Trabajo realizado con el apoyo del Programa UNAM-DGAPA-PAPIME, proyectos PE101019, PE101922 y PE106025.**

[![License: CC BY-SA 4.0](https://img.shields.io/badge/License-CC_BY--SA_4.0-lightgrey.svg)](https://creativecommons.org/licenses/by-sa/4.0/)

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import dfsol
import quadratic_func as qf

<a id="planteamiento"></a>
# Plateamiento.

Consideremos el siguiente sistema de ecuaciones:

$$
\left(
\begin{array}{cc}
 2 & -1 \\
-1 &  2
\end{array} \right)
\left(
\begin{array}{c}
x \\
y
\end{array} \right) =
\left(
\begin{array}{c}
638.89 \\ 
238.89
\end{array} \right) \tag{1}
$$

Este sistema proviene de la discretización de un problema de conducción de calor en 1D para dos nodos internos. Podemos generar el sistema usando la función `dfsol.sistema_lineal()` definida en el archivo [`dfsol.py`](dfsol.py). Posteriormente, se puede resolver el sistema con la función [`np.linalg.solve()`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.solve.html):

Observa el siguiente código:

In [2]:
# Definimos los parámetros físicos del problema
k = 1000
L = 0.5
TA = 500
TB = 100
q = 5e+6
N = 2

# Calculamos la matriz y el RHS del sistema
A, b = dfsol.sistema_lineal(TA, TB, q, L, k, N)

# Definimos el arreglo para almacenar la solución numérica
T = np.zeros(N+2)

# Valores de la tempertura en las fronteras
T[0]  = TA
T[-1] = TB

# Resolvemos el sistema lineal
T[1:-1] = np.linalg.solve(A, b)

# Imprimimos la matriz, el RHS y la solución
print(f"Matriz del sistema: \n{A}")
print(f"RHS: \n{b}")
print(f"Solución: \n{T}")

Matriz del sistema: 
[[ 2. -1.]
 [-1.  2.]]
RHS: 
[638.88888889 238.88888889]
Solución: 
[500.         505.55555556 372.22222222 100.        ]


## Forma cuadrática.
Existen métodos que permiten resolver este tipo de sistemas de una manera eficiente. Para ello se define lo que se conoce como la forma cuadrática del sistema de ecuaciones lineales, que se escribe como sigue:

$$ 
f(\mathbf{x}) = \dfrac{1}{2} \mathbf{x}^T A \mathbf{x} - \mathbf{x}^T \mathbf{b} + \mathbf{c} \tag{2}
$$

Para nuestro ejemplo tenemos que:

$$
A =
\left(
\begin{array}{cc}
 2 & -1 \\
-1 &  2
\end{array} \right),
\mathbf{x} =
\left(
\begin{array}{c}
x_{0} \\
x_{1}
\end{array} \right),
\mathbf{b} =
\left(
\begin{array}{c}
638.89 \\ 
238.89
\end{array}
\right), 
\mathbf{c} =
\left(
\begin{array}{c}
0\\ 0
\end{array}
\right), 
$$

Observa que en esta definición estamos usando $x_0 \equiv x$ y $x_1 \equiv y$.

Al ejecutar la siguiente celda veras un interactivo que muestra la función cuadrática de nuestro ejemplo, la solución del sistema lineal, junto con otros entes matemáticos.

In [3]:
quad_func = qf.QuadraticFunc(A, b, xlim = (100, 800), ylim = (-400, 1000), sizegrid = 40)
quad_func.interactive_qf()

Output()

En el interactivo observarás una gráfica de la función cuadrática en 3D que puedes rotar con el slider **Azimut**.
Puedes activar o desactivar los botones:
* **Solución** para visualizar la solución del sistema lineal,
* **Contornos** para visualizar los contornos de la función cuadrática. Observa que el mínimo se encuentra justo donde está la solución del sistema.
* **Eigenvectors** para visualizar los eigenvetores de la matriz, los cuales indican la dirección de los ejes de los contornos (que son elipses),
* **Gradiente** para visualizar los vectores que indican la dirección de máximo crecimiento de la función cuadrática.

Imagina que puedes poner un balín en algún lugar de la superficie 3D que representa la función cuadrática. Ese balín bajará rápidamente hacia el punto más bajo de la superficie (el valor mínimo) y ahí se quedará; este punto es la solución del sistema lineal. Esto se puede describir matemáticamente como se describe a continuación.

## Problema de minimización.

Si calculamos la derivada de la función cuadrática, obtenemos lo siguiente:

$$ f^\prime(\mathbf{x}) = \dfrac{1}{2} A^T \mathbf{x} + \dfrac{1}{2} A \mathbf{x} - \mathbf{b} \tag{3}$$

Cuando $A$ es **simétrica** se cumple que $A^T = A$, entonces la ecuación $(3)$ se simplifica a:

$$ f^\prime(\mathbf{x}) = A \mathbf{x} - \mathbf{b} \tag{4}$$

Supongamos que para $\mathbf{x} = \mathbf{x}_{sol}$ se cumple que $f^\prime(\mathbf{x}_{sol}) = 0$, por lo tanto:

$$
0 = A \mathbf{x}_{sol} - \mathbf{b} \Longrightarrow
A \mathbf{x}_{sol} = \mathbf{b}
$$

es decir, el vector $\mathbf{x}_{sol}$ será la solución del sistema lineal.

Recuerda que un punto crítico de $f(\mathbf{x})$ se obtiene cuando $ f^\prime(\mathbf{x}) = A \mathbf{x} - \mathbf{b} = 0$, es decir cuando $A \mathbf{x} = \mathbf{b}$.

Entonces, **encontrar el mínimo (o máximo) de la función cuadrática equivale a encontrar la solución del sistema lineal.**

Para llegar al mínimo, podemos comenzar el cualquier punto de la función cuadrática e ir dando pasos en la dirección del máximo descenso. Si recuerdas, el máximo crecimiento de una superficie está dado por el gradiente, entonces el máximo descenso será el *negativo* (dirección contraria) del gradiente.

Toda esta matemática es parte de los algoritmos que se conocen como **Métodos del subespacio de Krylov** y puedes ver una excelente explicación para comenzar con estos métodos en la siguiente referencia:


> Shewchuk, J. R. (1994). [An Introduction to the Conjugate Gradient Method Without the Agonizing Pain](https://www.cs.cmu.edu/~quake-papers/painless-conjugate-gradient.pdf). Carnegie-Mellon University. Department of Computer Science.

## Algoritmo de Gradiente Conjugado
Este algorimo tomando direcciones conjugadas optimizar las direcciones de descenso y reducir el número de pasos para obtener una solución. En este método se evita repetir pasos en una misma dirección.

<center>
<img src="./algo_gradiente_conjugado.png"  width='500px'/>
</center>

**Observaciones**.
* En este algoritmo se calcula el residuo inicialdel sistema definido por $\mathbf{r} = \mathbf{b} - A \mathbf{u}$. Esta será la primera dirección conjugada de búsqueda de la solución.
* Luego se calcula un tamaño de paso $\alpha$, para saber qué distancia se debe avanzar en la dirección conjugada.
* En la línea **7** se calcula una nueva aproximación de la solución.
* Posteriormente, en las líneas **8**, **9** y **10** se calcula la mueva dirección conjugada.

### Implementación.

En la siguiente celda se implementa el algoritmo gradiente conjugado. Revisa la implementación y comparala con el algoritmo. Ejecuta la celda para definir la función `conjugate_gradient()`.

In [4]:
def conjugate_gradient(A, b, tol, kmax, ux):
    # Solución inicial
    u = np.copy(ui)
    
    # Lista para los pasos intermedios
    xs = [u[0]]
    ys = [u[1]]
    
    # Cálculo y almacenamiento del residuo inicial
    r = b.T - A @ u
    res = np.linalg.norm(r, 2)
    res_list = [res]

    # Primera dirección conjugada
    d = r
    
    k = 0
    print("{:^2} {:^10} {:^12} {:^12}".format(" i ", "Residuo", "u0", "u1"))

    while(res > tol and k < kmax):
        # Tamaño del paso
        rk_old = r.T @ r
        alpha = rk_old / (d.T @ A @ d)

        # Nueva aproximación
        u = u + alpha * d
        xs.append(u[0])
        ys.append(u[1])

        # Cálculo y almacenamiento del residuo 
        r = r - alpha * A @ d
        res = np.linalg.norm(r, 2)
        res_list.append(res)

        # Cálculo de la nueva dirección conjugada
        beta = r.T @ r / rk_old
        d = r + beta * d
        
        k += 1
        
        print("{:2d} {:10.9f} ({:10.9f}, {:10.9f})".format(k, res, u[0], u[1]))
    return u, xs, ys, res_list, k 


Observa que la función `steepest()` regresa 5 valores: 

* `u` la solución obtenida, 
* `xs` y `ys` componentes de las soluciones aproximadas en cada paso, 
* `res_list` una lista con el residuo para cada iteración.
* `k` el número total de iteraciones realizadas.

<div class="alert alert-info">

## Ejemplo 1.

Haciendo uso de la función `conjugate_gradient()` definida en la celda anterior, aproximaremos la solución del sistema de ecuaciones definido en la sección <a href="#planteamiento">Planteamiento</a>, utilizando los siguientes datos:

* Solución inicial `ui =` $(100, 100)$.
* Tolerancia `tol` = $1 \times 10^{-5}$.
* Número máximo de iteraciones `kmax` = $50$.

Posteriormente se realizan algunas gráficas.
 
</div>

In [5]:
ui = np.array([100, 100])
tol = 1e-5
kmax = 50

# Método CGM
solCG, xs, ys, res, it = conjugate_gradient(A, b, tol, kmax, ui)

 i   Residuo        u0           u1     
 1 321.217568692 (455.316803201, 191.576495670)
 2 0.000000000 (505.555555556, 372.222222222)


In [6]:
quad_func.set_steps(xs, ys) # Pasos calculados
quad_func.interactive_steps() # Ejecución del interactivo

Output()

<div class="alert alert-success">

**Pregunta**.
¿Qué crees que pasará si cambiamos la solución inicial (punto amarillo)?

</div>

<div class="alert alert-info">

Graficamos el residuo que se va obteniendo en cada paso del algoritmo.
</div>

In [ ]:
# Gráfica del residuo
plt.figure(figsize=(6,3))
plt.title("Residuo", loc="left")
plt.plot(list(range(0,it+1)), res, marker=".", label="Gradiente conjugado")
plt.xlabel("Iteraciones")
plt.yscale("log")
plt.legend()
plt.grid(True)
plt.show()

Almacenamos el error en el archivo `zd_residuoCG.npy` y los pasos en el archivo `zd_pasosCG.npy`.

In [ ]:
dir_data = Path("datos")
dir_data.mkdir(parents=True, exist_ok=True)

np.save("./datos/zd_residuoCG.npy", res)
np.save("./datos/zd_pasosCG.npy", np.array([xs,ys]))

Checamos que se pueden leer los archivos con la función `np.load()`:

In [ ]:
print("Residuo = \n{}\n".format(np.load("./datos/zd_residuoCG.npy")))
print("(xs, ys) = \n{}\n".format(np.load("./datos/zd_pasosCG.npy")))